No contextual Features

In [ ]:
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoModel, AutoTokenizer
from peft import LoraConfig, get_peft_model
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier

def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

device = get_device()
print("Using device:", device)

file_path = 'cleaned_Python.csv'
data = pd.read_csv(file_path)

data['concatenated_text'] = data['Processed Body'] + " " + data['Tags'] + " " + data['Title']

sbert_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2").to(device)

codebert_model_name = "microsoft/codebert-base-mlm"
tokenizer = AutoTokenizer.from_pretrained(codebert_model_name)
base_model = AutoModel.from_pretrained(codebert_model_name)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["query", "value", "key", "dense"]
)

codebert_model = get_peft_model(base_model, lora_config).to(device)

def extract_sbert_embeddings(texts, batch_size=64):
    embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        batch_embeddings = sbert_model.encode(batch, convert_to_numpy=True, device=device)
        embeddings.append(batch_embeddings)
    return np.vstack(embeddings)

def extract_codebert_embeddings(texts, batch_size=64):
    embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        inputs = tokenizer(batch, padding=True, truncation=True, return_tensors="pt", max_length=512).to(device)
        with torch.no_grad():
            outputs = codebert_model(**inputs)
        batch_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        embeddings.append(batch_embeddings)
    return np.vstack(embeddings)

print("Extracting Sentence-BERT embeddings...")
sbert_embeddings = extract_sbert_embeddings(data['concatenated_text'].tolist())
print("Extracting CodeBERT embeddings...")
codebert_embeddings = extract_codebert_embeddings(data['concatenated_text'].tolist())

combined_features = np.hstack((sbert_embeddings, codebert_embeddings))

label_mapping = {'Basic': 0, 'Intermediate': 1, 'Advanced': 2}
labels = data['Label'].map(label_mapping).values

X_train, X_test, y_train, y_test = train_test_split(combined_features, labels, test_size=0.2, random_state=42, stratify=labels)

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

clf = RandomForestClassifier(n_estimators=400, max_depth=20, bootstrap=True, max_features='sqrt',
                             min_samples_split=10, min_samples_leaf=1, random_state=42)
clf.fit(X_train_resampled, y_train_resampled)

y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred, target_names=label_mapping.keys(), digits=4))


Using device: cuda
Extracting Sentence-BERT embeddings...
Extracting CodeBERT embeddings...
              precision    recall  f1-score   support

       Basic     0.5152    0.3864    0.4416        44
Intermediate     0.6099    0.8037    0.6935       107
    Advanced     0.6500    0.3023    0.4127        43

    accuracy                         0.5979       194
   macro avg     0.5917    0.4975    0.5159       194
weighted avg     0.5973    0.5979    0.5741       194



SBERT with LoRA

In [ ]:
!pip install sentence_transformers
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoModel, AutoTokenizer
from peft import LoraConfig, get_peft_model
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

file_path = 'Java1.csv'
data = pd.read_csv(file_path)


numerical_cols = [
    'User_Reputation', 'Bronze_badge', 'Gold_badge', 'Silver_badge',
    'User Id', 'Accepted Answer ID', 'View Count', 'Answer Count',
    'Score', 'Interval from first', 'Interval from accepted',
    'Total count urls and imgs', 'LOC', 'Question_Length'
]

imputer = SimpleImputer(strategy='mean')
data[numerical_cols] = imputer.fit_transform(data[numerical_cols])
scaler = StandardScaler()
data[numerical_cols] = scaler.fit_transform(data[numerical_cols])

model_name = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModel.from_pretrained(model_name)

lora_config = LoraConfig(
    r=8,  
    lora_alpha=16,  
    lora_dropout=0.1,  
    target_modules=["query", "value"]
)

model = get_peft_model(base_model, lora_config)
model.to(device)
print(model)

def extract_embeddings(texts):
    inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt", max_length=512).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state[:, 0, :].cpu().numpy()  

data['concatenated_text'] = data['Processed Body'] + " " + data['Tags'] + " " + data['Title']

text_embeddings = extract_embeddings(data['concatenated_text'].tolist())

numerical_features = data[numerical_cols].values
combined_features = np.hstack((text_embeddings, numerical_features))

label_mapping = {'Basic': 0, 'Intermediate': 1, 'Advanced': 2}
labels = data['Label'].map(label_mapping).values

X_train, X_test, y_train, y_test = train_test_split(combined_features, labels, test_size=0.2, random_state=42, stratify=labels)

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

clf = RandomForestClassifier(n_estimators=400, max_depth=20, bootstrap=True, max_features='sqrt',
                             min_samples_split=10, min_samples_leaf=1, random_state=42)
clf.fit(X_train_resampled, y_train_resampled)

y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred, target_names=label_mapping.keys(), digits=4))


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Using device: cuda
PeftModel(
  (base_model): LoraModel(
    (model): BertModel(
      (embeddings): BertEmbeddings(
        (word_embeddings): Embedding(30522, 384, padding_idx=0)
        (position_embeddings): Embedding(512, 384)
        (token_type_embeddings): Embedding(2, 384)
        (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (encoder): BertEncoder(
        (layer): ModuleList(
          (0-5): 6 x BertLayer(
            (attention): BertAttention(
              (self): BertSdpaSelfAttention(
                (query): lora.Linear(
                  (base_layer): Linear(in_features=384, out_features=384, bias=True)
                  (lora_dropout): ModuleDict(
                    (default): Dropout(p=0.1, inplace=False)
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=384, out_features=8, bias=False)
                  )
                

Only CodeBERT

In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoModel, AutoTokenizer
from peft import LoraConfig, get_peft_model
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier

def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

device = get_device()
print("Using device:", device)

file_path = 'cleaned_Python.csv'
data = pd.read_csv(file_path)

data['concatenated_text'] = data['Processed Body'] + " " + data['Tags'] + " " + data['Title']

codebert_model_name = "microsoft/codebert-base-mlm"
tokenizer = AutoTokenizer.from_pretrained(codebert_model_name)
base_model = AutoModel.from_pretrained(codebert_model_name)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["query", "value", "key", "dense"]
)

codebert_model = get_peft_model(base_model, lora_config).to(device)

def extract_codebert_embeddings(texts, batch_size=64):
    embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        inputs = tokenizer(batch, padding=True, truncation=True, return_tensors="pt", max_length=512).to(device)
        with torch.no_grad():
            outputs = codebert_model(**inputs)
        batch_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        embeddings.append(batch_embeddings)
    return np.vstack(embeddings)

print("Extracting CodeBERT embeddings...")
codebert_embeddings = extract_codebert_embeddings(data['concatenated_text'].tolist())

combined_features = codebert_embeddings

label_mapping = {'Basic': 0, 'Intermediate': 1, 'Advanced': 2}
labels = data['Label'].map(label_mapping).values

X_train, X_test, y_train, y_test = train_test_split(combined_features, labels, test_size=0.2, random_state=42, stratify=labels)

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

clf = RandomForestClassifier(n_estimators=400, max_depth=20, bootstrap=True, max_features='sqrt',
                             min_samples_split=10, min_samples_leaf=1, random_state=42)
clf.fit(X_train_resampled, y_train_resampled)

y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred, target_names=label_mapping.keys(), digits=4))


Using device: cuda
Extracting CodeBERT embeddings...
              precision    recall  f1-score   support

       Basic     0.5143    0.4091    0.4557        44
Intermediate     0.6058    0.7757    0.6803       107
    Advanced     0.5909    0.3023    0.4000        43

    accuracy                         0.5876       194
   macro avg     0.5703    0.4957    0.5120       194
weighted avg     0.5818    0.5876    0.5672       194



Using KeyBERT with Sentence BERT

In [ ]:
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from keybert import KeyBERT
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier

def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

device = get_device()
print("Using device:", device)

file_path = 'cleaned_Python.csv'
data = pd.read_csv(file_path)

data['concatenated_text'] = data['Processed Body'] + " " + data['Tags'] + " " + data['Title']

kw_model = KeyBERT("sentence-transformers/all-MiniLM-L6-v2")
data['keywords'] = data['concatenated_text'].apply(lambda x: " ".join([kw[0] for kw in kw_model.extract_keywords(x, top_n=5)]))

sbert_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2").to(device)

def extract_sbert_embeddings(texts, batch_size=64):
    embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        batch_embeddings = sbert_model.encode(batch, convert_to_numpy=True, device=device)
        embeddings.append(batch_embeddings)
    return np.vstack(embeddings)

print("Extracting Sentence-BERT embeddings from keywords...")
sbert_embeddings = extract_sbert_embeddings(data['keywords'].tolist())

combined_features = sbert_embeddings

label_mapping = {'Basic': 0, 'Intermediate': 1, 'Advanced': 2}
labels = data['Label'].map(label_mapping).values

X_train, X_test, y_train, y_test = train_test_split(combined_features, labels, test_size=0.2, random_state=42, stratify=labels)

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

clf = RandomForestClassifier(n_estimators=400, max_depth=20, bootstrap=True, max_features='sqrt',
                             min_samples_split=10, min_samples_leaf=1, random_state=42)
clf.fit(X_train_resampled, y_train_resampled)

y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred, target_names=label_mapping.keys(), digits=4))


Using device: cuda
Extracting Sentence-BERT embeddings from keywords...
              precision    recall  f1-score   support

       Basic     0.2381    0.1136    0.1538        44
Intermediate     0.5425    0.7757    0.6385       107
    Advanced     0.3500    0.1628    0.2222        43

    accuracy                         0.4897       194
   macro avg     0.3769    0.3507    0.3382       194
weighted avg     0.4308    0.4897    0.4363       194



With TF-IDF

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier

file_path = 'cleaned_Python.csv'
data = pd.read_csv(file_path)

data['concatenated_text'] = data['Processed Body'] + " " + data['Tags'] + " " + data['Title']

vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X_tfidf = vectorizer.fit_transform(data['concatenated_text']).toarray()

label_mapping = {'Basic': 0, 'Intermediate': 1, 'Advanced': 2}
labels = data['Label'].map(label_mapping).values

X_train, X_test, y_train, y_test = train_test_split(X_tfidf, labels, test_size=0.2, random_state=42, stratify=labels)

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

clf = RandomForestClassifier(n_estimators=400, max_depth=20, bootstrap=True, max_features='sqrt',
                             min_samples_split=10, min_samples_leaf=1, random_state=42)
clf.fit(X_train_resampled, y_train_resampled)

y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred, target_names=label_mapping.keys(), digits=4))


              precision    recall  f1-score   support

       Basic     0.4286    0.4773    0.4516        44
Intermediate     0.6015    0.7477    0.6667       107
    Advanced     0.6667    0.1860    0.2909        43

    accuracy                         0.5619       194
   macro avg     0.5656    0.4703    0.4697       194
weighted avg     0.5767    0.5619    0.5346       194



: 

CODEBERT WITH LORA

In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from peft import LoraConfig, get_peft_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

df = pd.read_csv("Java1.csv")

df['concatenated_text'] = df['Processed Body'] + " " + df['Tags'] + " " + df['Title']

label_encoder = LabelEncoder()
df['Label'] = label_encoder.fit_transform(df['Label'])

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['concatenated_text'], df['Label'], test_size=0.2, random_state=42
)

model_name = "microsoft/codebert-base-mlm"
tokenizer = AutoTokenizer.from_pretrained(model_name)

base_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(label_encoder.classes_))

lora_config = LoraConfig(
    r=8,  
    lora_alpha=16,  
    lora_dropout=0.1,  
    target_modules=["query", "value"]  
)

model = get_peft_model(base_model, lora_config)
model.to(device) 

print(model)

train_encodings = tokenizer(
    train_texts.tolist(), truncation=True, padding=True, return_tensors="pt", max_length=512
)
test_encodings = tokenizer(
    test_texts.tolist(), truncation=True, padding=True, return_tensors="pt", max_length=512
)

train_dataset = TensorDataset(
    train_encodings["input_ids"].to(device),
    train_encodings["attention_mask"].to(device),
    torch.tensor(train_labels.tolist()).to(device),
)
test_dataset = TensorDataset(
    test_encodings["input_ids"].to(device),
    test_encodings["attention_mask"].to(device),
    torch.tensor(test_labels.tolist()).to(device),
)

batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
loss_fn = torch.nn.CrossEntropyLoss()

epochs = 15
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch in train_loader:
        input_ids, attention_mask, labels = batch
        input_ids, attention_mask, labels = input_ids.to(device), attention_mask.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss:.4f}")

model.eval()
predictions = []
true_labels = []
confidence_scores = []  
with torch.no_grad():
    for batch in test_loader:
        input_ids, attention_mask, labels = batch
        input_ids, attention_mask, labels = input_ids.to(device), attention_mask.to(device), labels.to(device)

        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        probabilities = torch.softmax(logits, dim=1)
        max_probabilities, predicted = torch.max(probabilities, dim=1)

        predictions.extend(predicted.cpu().tolist())
        true_labels.extend(labels.cpu().tolist())
        confidence_scores.extend(max_probabilities.cpu().tolist())

accuracy = accuracy_score(true_labels, predictions)
print("Accuracy:", accuracy)


Using device: cuda


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at microsoft/codebert-base-mlm and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


PeftModel(
  (base_model): LoraModel(
    (model): RobertaForSequenceClassification(
      (roberta): RobertaModel(
        (embeddings): RobertaEmbeddings(
          (word_embeddings): Embedding(50265, 768, padding_idx=1)
          (position_embeddings): Embedding(514, 768, padding_idx=1)
          (token_type_embeddings): Embedding(1, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (encoder): RobertaEncoder(
          (layer): ModuleList(
            (0-11): 12 x RobertaLayer(
              (attention): RobertaAttention(
                (self): RobertaSdpaSelfAttention(
                  (query): lora.Linear(
                    (base_layer): Linear(in_features=768, out_features=768, bias=True)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.1, inplace=False)
                    )
                    (lora_A): ModuleDict(
                

SBERT WITH LORA FINETUNED THEN EMBEDDINGS EXTRACTED AND THEN COMBINED WITH CONTEXTUAL FEATURES THEN RF CLASSIFEID

In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType
from sentence_transformers import SentenceTransformer, losses
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE


In [ ]:
file_path = 'Java1.csv'
data = pd.read_csv(file_path)

numerical_cols = [
    'User_Reputation', 'Bronze_badge', 'Gold_badge', 'Silver_badge',
    'User Id', 'Accepted Answer ID', 'View Count', 'Answer Count',
    'Score', 'Interval from first', 'Interval from accepted',
    'Total count urls and imgs', 'LOC', 'Question_Length'
]
imputer = SimpleImputer(strategy='mean')
data[numerical_cols] = imputer.fit_transform(data[numerical_cols])
scaler = StandardScaler()
data[numerical_cols] = scaler.fit_transform(data[numerical_cols])

data['concatenated_text'] = data[['Processed Body', 'Tags', 'Title']].fillna('').agg(' '.join, axis=1)

label_mapping = {'Basic': 0, 'Intermediate': 1, 'Advanced': 2}
labels = data['Label'].map(label_mapping).values


In [ ]:
from transformers import AutoModel, AutoTokenizer
from peft import LoraConfig, get_peft_model
import torch.nn as nn
import torch

model_name = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModel.from_pretrained(model_name)

lora_config = LoraConfig(
    r=8,  
    lora_alpha=16, 
    lora_dropout=0.1,  
    target_modules=["query", "value"],  
)

model = get_peft_model(base_model, lora_config)


In [ ]:
class SBERTWithClassification(nn.Module):
    def __init__(self, base_model, num_classes):
        super(SBERTWithClassification, self).__init__()
        self.base_model = base_model
        self.classifier = nn.Linear(384, num_classes)  

    def forward(self, input_ids, attention_mask):
        outputs = self.base_model(input_ids=input_ids, attention_mask=attention_mask)
        cls_embedding = outputs.last_hidden_state[:, 0, :]  
        logits = self.classifier(cls_embedding)
        return logits

num_classes = 3 
model = SBERTWithClassification(model, num_classes)
model.to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))


SBERTWithClassification(
  (base_model): PeftModel(
    (base_model): LoraModel(
      (model): BertModel(
        (embeddings): BertEmbeddings(
          (word_embeddings): Embedding(30522, 384, padding_idx=0)
          (position_embeddings): Embedding(512, 384)
          (token_type_embeddings): Embedding(2, 384)
          (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (encoder): BertEncoder(
          (layer): ModuleList(
            (0-5): 6 x BertLayer(
              (attention): BertAttention(
                (self): BertSdpaSelfAttention(
                  (query): lora.Linear(
                    (base_layer): Linear(in_features=384, out_features=384, bias=True)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.1, inplace=False)
                    )
                    (lora_A): ModuleDict(
                      (default): Linear(in_features=384, 

In [ ]:
class SOQuestionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        inputs = self.tokenizer(self.texts[idx], padding="max_length", truncation=True, max_length=self.max_length, return_tensors="pt")
        input_ids = inputs["input_ids"].squeeze(0)
        attention_mask = inputs["attention_mask"].squeeze(0)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return {"input_ids": input_ids, "attention_mask": attention_mask, "label": label}

X_train, X_test, y_train, y_test = train_test_split(data['concatenated_text'].tolist(), labels, test_size=0.2, random_state=42, stratify=labels)

train_dataset = SOQuestionDataset(X_train, y_train, tokenizer)
test_dataset = SOQuestionDataset(X_test, y_test, tokenizer)

train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=16, shuffle=False)


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
loss_fn = nn.CrossEntropyLoss()

epochs = 3
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.train()
for epoch in range(epochs):
    total_loss = 0
    for batch in train_dataloader:
        optimizer.zero_grad()
        input_ids, attention_mask, labels = batch["input_ids"].to(device), batch["attention_mask"].to(device), batch["label"].to(device)
        
        logits = model(input_ids=input_ids, attention_mask=attention_mask)  
        loss = loss_fn(logits, labels)  
        
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1} Loss: {total_loss / len(train_dataloader)}")


Epoch 1 Loss: 1.016206071061908
Epoch 2 Loss: 0.9898616502869804
Epoch 3 Loss: 0.9631311353647484


In [ ]:
!pip install sentence_transformers peft transformers scikit-learn imbalanced-learn

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sentence_transformers import SentenceTransformer
from transformers import AutoModel, AutoTokenizer
from peft import LoraConfig, get_peft_model
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

file_path = 'Java1.csv'
data = pd.read_csv(file_path)

numerical_cols = [
    'User_Reputation', 'Bronze_badge', 'Gold_badge', 'Silver_badge',
    'User Id', 'Accepted Answer ID', 'View Count', 'Answer Count',
    'Score', 'Interval from first', 'Interval from accepted',
    'Total count urls and imgs', 'LOC', 'Question_Length'
]

imputer = SimpleImputer(strategy='mean')
data[numerical_cols] = imputer.fit_transform(data[numerical_cols])
scaler = StandardScaler()
data[numerical_cols] = scaler.fit_transform(data[numerical_cols])

model_name = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModel.from_pretrained(model_name)

lora_config = LoraConfig(
    r=8,  
    lora_alpha=16,  
    lora_dropout=0.1,  
    target_modules=["query", "value"]
)

model = get_peft_model(base_model, lora_config)
model.to(device)

data['concatenated_text'] = data['Processed Body'] + " " + data['Tags'] + " " + data['Title']

class StackOverflowDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        inputs = tokenizer(
            self.texts[idx], 
            padding="max_length", 
            truncation=True, 
            return_tensors="pt", 
            max_length=512
        )

        if "token_type_ids" in inputs:
            del inputs["token_type_ids"]

        return {
            "input_ids": inputs["input_ids"].squeeze(0),
            "attention_mask": inputs["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.long)
        }

label_mapping = {'Basic': 0, 'Intermediate': 1, 'Advanced': 2}
labels = data['Label'].map(label_mapping).values




X_train, X_test, y_train, y_test = train_test_split(
    data['concatenated_text'].tolist(), labels, test_size=0.2, random_state=42, stratify=labels
)

train_dataset = StackOverflowDataset(X_train, y_train)
test_dataset = StackOverflowDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

class SBERTWithClassification(nn.Module):
    def __init__(self, base_model, num_classes):
        super(SBERTWithClassification, self).__init__()
        self.base_model = base_model
        self.classifier = nn.Linear(384, num_classes)  # SBERT output dim is 384

    def forward(self, input_ids, attention_mask):
        outputs = self.base_model(input_ids=input_ids, attention_mask=attention_mask)
        cls_embedding = outputs.last_hidden_state[:, 0, :]  # Extract CLS token
        logits = self.classifier(cls_embedding)
        return logits

num_classes = len(label_mapping)
classifier_model = SBERTWithClassification(model, num_classes).to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = optim.AdamW(classifier_model.parameters(), lr=5e-5)

num_epochs = 3
for epoch in range(num_epochs):
    classifier_model.train()
    total_loss = 0

    for batch in train_loader:
        optimizer.zero_grad()
        input_ids, attention_mask, labels = (
            batch["input_ids"].to(device),
            batch["attention_mask"].to(device),
            batch["label"].to(device)
        )
        outputs = classifier_model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {total_loss/len(train_loader):.4f}")

def extract_embeddings(texts):
    inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt", max_length=512).to(device)

    if "token_type_ids" in inputs:
        del inputs["token_type_ids"]

    with torch.no_grad():
        outputs = classifier_model.base_model(**inputs)
    
    return outputs.last_hidden_state[:, 0, :].cpu().numpy()  # Extract CLS token embeddings

text_embeddings = np.array([extract_embeddings([text])[0] for text in data['concatenated_text'].tolist()])

numerical_features = data[numerical_cols].values
combined_features = np.hstack((text_embeddings, numerical_features))

labels = data['Label'].values  

print("Shape of text embeddings:", text_embeddings.shape) 
print("Shape of numerical features:", numerical_features.shape) 
print("Shape of labels before reshaping:", labels.shape)  

labels = labels.reshape(-1) 
print("Shape of labels after reshaping:", labels.shape)  

assert text_embeddings.shape[0] == numerical_features.shape[0] == labels.shape[0], "Mismatch in dataset sizes!"

X_train, X_test, y_train, y_test = train_test_split(combined_features, labels, test_size=0.20, random_state=42, stratify=labels)

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

clf = RandomForestClassifier(n_estimators=400, max_depth=20, bootstrap=True, max_features='sqrt',
                             min_samples_split=10, min_samples_leaf=1, random_state=42)
clf.fit(X_train_resampled, y_train_resampled)

y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred, target_names=label_mapping.keys(), digits =4))


Using device: cuda
Epoch 1/3, Loss: 0.9979
Epoch 2/3, Loss: 0.9418
Epoch 3/3, Loss: 0.8795
Shape of text embeddings: (1042, 384)
Shape of numerical features: (1042, 14)
Shape of labels before reshaping: (1042,)
Shape of labels after reshaping: (1042,)
              precision    recall  f1-score   support

       Basic     0.3333    0.2500    0.2857         4
Intermediate     0.7692    0.9375    0.8451        32
    Advanced     0.7273    0.4706    0.5714        17

    accuracy                         0.7358        53
   macro avg     0.6099    0.5527    0.5674        53
weighted avg     0.7229    0.7358    0.7151        53



In [ ]:
labels = data['Label'].values 


print("Shape of text embeddings:", text_embeddings.shape) 
print("Shape of numerical features:", numerical_features.shape)  
print("Shape of labels before reshaping:", labels.shape)  

labels = labels.reshape(-1)  
print("Shape of labels after reshaping:", labels.shape)  

assert text_embeddings.shape[0] == numerical_features.shape[0] == labels.shape[0], "Mismatch in dataset sizes!"

X_train, X_test, y_train, y_test = train_test_split(combined_features, labels, test_size=0.20, random_state=42, stratify=labels)

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

clf = RandomForestClassifier(n_estimators=400, max_depth=20, bootstrap=True, max_features='sqrt',
                             min_samples_split=10, min_samples_leaf=1, random_state=42)
clf.fit(X_train_resampled, y_train_resampled)

y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred, target_names=label_mapping.keys(), digits =4))


Shape of text embeddings: (1042, 384)
Shape of numerical features: (1042, 14)
Shape of labels before reshaping: (1042,)
Shape of labels after reshaping: (1042,)
              precision    recall  f1-score   support

       Basic     0.5000    0.4167    0.4545        12
Intermediate     0.7879    0.8387    0.8125        93
    Advanced     0.6042    0.5577    0.5800        52

    accuracy                         0.7134       157
   macro avg     0.6307    0.6044    0.6157       157
weighted avg     0.7050    0.7134    0.7081       157



In [ ]:
num_epochs = 10
for epoch in range(num_epochs):
    classifier_model.train()
    total_loss = 0

    for batch in train_loader:
        optimizer.zero_grad()
        input_ids, attention_mask, labels = (
            batch["input_ids"].to(device),
            batch["attention_mask"].to(device),
            batch["label"].to(device)
        )
        outputs = classifier_model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {total_loss/len(train_loader):.4f}")

def extract_embeddings(texts):
    inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt", max_length=512).to(device)

    if "token_type_ids" in inputs:
        del inputs["token_type_ids"]

    with torch.no_grad():
        outputs = classifier_model.base_model(**inputs)
    
    return outputs.last_hidden_state[:, 0, :].cpu().numpy()  

text_embeddings = np.array([extract_embeddings([text])[0] for text in data['concatenated_text'].tolist()])

numerical_features = data[numerical_cols].values
combined_features = np.hstack((text_embeddings, numerical_features))

labels = data['Label'].values  

print("Shape of text embeddings:", text_embeddings.shape)  
print("Shape of numerical features:", numerical_features.shape)  
print("Shape of labels before reshaping:", labels.shape)  

labels = labels.reshape(-1)  
print("Shape of labels after reshaping:", labels.shape)  

assert text_embeddings.shape[0] == numerical_features.shape[0] == labels.shape[0], "Mismatch in dataset sizes!"

X_train, X_test, y_train, y_test = train_test_split(combined_features, labels, test_size=0.20, random_state=42, stratify=labels)

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

clf = RandomForestClassifier(n_estimators=400, max_depth=20, bootstrap=True, max_features='sqrt',
                             min_samples_split=10, min_samples_leaf=1, random_state=42)
clf.fit(X_train_resampled, y_train_resampled)

y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred, target_names=label_mapping.keys(), digits =4))

Epoch 1/10, Loss: 0.6386
Epoch 2/10, Loss: 0.6362
Epoch 3/10, Loss: 0.6254
Epoch 4/10, Loss: 0.6195
Epoch 5/10, Loss: 0.6227
Epoch 6/10, Loss: 0.5993
Epoch 7/10, Loss: 0.6171
Epoch 8/10, Loss: 0.5945
Epoch 9/10, Loss: 0.5876
Epoch 10/10, Loss: 0.5795
Shape of text embeddings: (1042, 384)
Shape of numerical features: (1042, 14)
Shape of labels before reshaping: (1042,)
Shape of labels after reshaping: (1042,)
              precision    recall  f1-score   support

       Basic     0.6250    0.4167    0.5000        12
Intermediate     0.7864    0.8710    0.8265        93
    Advanced     0.6304    0.5577    0.5918        52

    accuracy                         0.7325       157
   macro avg     0.6806    0.6151    0.6395       157
weighted avg     0.7224    0.7325    0.7238       157



In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, accuracy_score
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
import numpy as np

k = 5
skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)

all_accuracy = []
all_reports = []

for fold, (train_idx, test_idx) in enumerate(skf.split(combined_features, labels), 1):
    print(f"\n🔁 Fold {fold}/{k}")

    X_train, X_test = combined_features[train_idx], combined_features[test_idx]
    y_train, y_test = labels[train_idx], labels[test_idx]

    smote = SMOTE(random_state=42)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

    clf = RandomForestClassifier(
        n_estimators=400,
        max_depth=20,
        bootstrap=True,
        max_features='sqrt',
        min_samples_split=10,
        min_samples_leaf=1,
        random_state=42
    )
    clf.fit(X_train_resampled, y_train_resampled)

    y_pred = clf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    all_accuracy.append(acc)
    
    report = classification_report(y_test, y_pred, target_names=label_mapping.keys(), digits=4, output_dict=True)
    all_reports.append(report)
    
    print(f"✅ Accuracy for fold {fold}: {acc*100:.2f}%")

mean_accuracy = np.mean(all_accuracy)
std_accuracy = np.std(all_accuracy)
print(f"\n📊 Mean Accuracy over {k} folds: {mean_accuracy*100:.2f}% ± {std_accuracy*100:.2f}%")



🔁 Fold 1/5
✅ Accuracy for fold 1: 71.29%

🔁 Fold 2/5
✅ Accuracy for fold 2: 73.68%

🔁 Fold 3/5
✅ Accuracy for fold 3: 75.48%

🔁 Fold 4/5
✅ Accuracy for fold 4: 73.56%

🔁 Fold 5/5
✅ Accuracy for fold 5: 74.04%

📊 Mean Accuracy over 5 folds: 73.61% ± 1.35%
